# 추가 백엔드-4 (Vision 반복 FAIL step_description) - Jupyter Notebook

사양서 기준 구현 요약

- 입력: `prod_day` (YYYYMMDD)
- 산출: `prod_day day` + `prod_day night` **둘 다** 산출
- 대상 테이블: `a3_vision_table.vision_table` (result='FAIL')
- 반복 기준: 동일 `barcode_information` + 동일 `step_description` (station 제외)
- Dedup: (`barcode_information`, `step_description`, `end_day`, `end_time`) 완전 동일 row는 1건으로 dedup
- pn 분류: `g_production_film.remark_info`를 조회하여 **barcode_information의 18번째 문자**로 pn 매핑 (없으면 `Unknown`)
- 출력: 1회/2회/3회+ 각각 별도 DataFrame (day/night 각각)


In [1]:
# =========================
# 0) 설정
# =========================
from __future__ import annotations

from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
from typing import Dict, Tuple, Optional

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

# (1) 입력: 생산일 (WINDOW 기준)
prod_day = "20260130"   # <-- 여기만 바꾸면 됨 (YYYYMMDD)

# (2) DB 설정 (사양서)
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

# (3) 대상 테이블
VISION_SCHEMA = "a3_vision_table"
VISION_TABLE  = "vision_table"

REMARK_SCHEMA = "g_production_film"
REMARK_TABLE  = "remark_info"

def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

def yyyymmdd_to_date(s: str) -> date:
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))

def date_to_yyyymmdd(d: date) -> str:
    return d.strftime("%Y%m%d")

def shift_window(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    """사양서 기준 KST window
    - day:   D 08:30:00 ~ D 20:29:59
    - night: D 20:30:00 ~ D+1 08:29:59
    """
    d = yyyymmdd_to_date(prod_day)
    if shift_type == "day":
        start = datetime.combine(d, time(8, 30, 0), tzinfo=KST)
        end   = datetime.combine(d, time(20, 29, 59), tzinfo=KST)
    elif shift_type == "night":
        start = datetime.combine(d, time(20, 30, 0), tzinfo=KST)
        end   = datetime.combine(d + timedelta(days=1), time(8, 29, 59), tzinfo=KST)
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return start, end

def get_barcode_key_18th(barcode: Optional[str]) -> Optional[str]:
    """barcode_information 18번째 문자(1-indexed) -> python index 17"""
    if not barcode:
        return None
    if len(barcode) < 18:
        return None
    return barcode[17]

engine = make_engine()


In [2]:
# =========================
# 1) remark_info 조회 -> pn 매핑(dict)
# =========================
remark_sql = text(f"""
SELECT
  barcode_information,
  pn
FROM {REMARK_SCHEMA}.{REMARK_TABLE}
""")

with engine.begin() as conn:
    remark_df = pd.read_sql(remark_sql, conn)

# remark_info.barcode_information은 18번째 문자 key 값(예: 'J')
pn_map: Dict[str, str] = (
    remark_df.dropna(subset=["barcode_information", "pn"])
             .assign(barcode_information=lambda d: d["barcode_information"].astype(str).str.strip())
             .set_index("barcode_information")["pn"]
             .to_dict()
)

remark_df


,barcode_information,pn
0,C,35643009
1,P,35643010
2,1,35654264
3,N,35749091
4,J,35930927
5,S,35930928


In [3]:
# =========================
# 2) vision_table 조회 (shift별) + dedup
# =========================
def fetch_fail_rows(prod_day: str, shift_type: str) -> pd.DataFrame:
    d0 = yyyymmdd_to_date(prod_day)
    d1 = d0 + timedelta(days=1)
    d0s, d1s = date_to_yyyymmdd(d0), date_to_yyyymmdd(d1)

    if shift_type == "day":
        where_sql = """
            end_day = :d0
            AND end_time >= '08:30:00'
            AND end_time <= '20:29:59'
        """
        params = {"d0": d0s}
    else:
        where_sql = """
            (
                (end_day = :d0 AND end_time >= '20:30:00' AND end_time <= '23:59:59')
                OR
                (end_day = :d1 AND end_time >= '00:00:00' AND end_time <= '08:29:59')
            )
        """
        params = {"d0": d0s, "d1": d1s}

    # Dedup: (barcode_information, step_description, end_day, end_time) 완전 동일 row 1개만
    sql = text(f"""
        SELECT DISTINCT ON (barcode_information, step_description, end_day, end_time)
            end_day,
            end_time,
            barcode_information,
            step_description
        FROM {VISION_SCHEMA}.{VISION_TABLE}
        WHERE
            result = 'FAIL'
            AND {where_sql}
        ORDER BY
            barcode_information, step_description, end_day, end_time
    """)

    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=params)

    # pn 매핑
    df["barcode_key_18th"] = df["barcode_information"].apply(get_barcode_key_18th)
    df["pn"] = df["barcode_key_18th"].map(pn_map).fillna("Unknown")

    # 메타
    df["prod_day"] = prod_day
    df["shift_type"] = shift_type
    return df

day_raw = fetch_fail_rows(prod_day, "day")
night_raw = fetch_fail_rows(prod_day, "night")

day_raw.head(), night_raw.head()


(    end_day  end_time             barcode_information step_description  \
 0  20260130  18:45:37  BA1WJ26028500020SJ8T-14F014-AF       intensity1   
 1  20260130  18:45:37  BA1WJ26028500020SJ8T-14F014-AF       intensity7   
 2  20260130  13:07:01  BA1WJ26028504043SJ8T-14F014-AF       intensity1   
 3  20260130  13:07:01  BA1WJ26028504043SJ8T-14F014-AF       intensity2   
 4  20260130  13:07:01  BA1WJ26028504043SJ8T-14F014-AF       intensity3   
 
   barcode_key_18th        pn  prod_day shift_type  
 0                J  35930927  20260130        day  
 1                J  35930927  20260130        day  
 2                J  35930927  20260130        day  
 3                J  35930927  20260130        day  
 4                J  35930927  20260130        day  ,
     end_day  end_time              barcode_information step_description  \
 0  20260130  20:38:48  BA1WJ26030501090USJ8T-14F014-AF       intensity1   
 1  20260130  20:38:48  BA1WJ26030501090USJ8T-14F014-AF       intensity7   
 

In [4]:
# =========================
# 3) 반복 횟수 계산 -> 1회/2회/3회+ 분리 집계
# =========================
def build_repeat_dfs(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """raw: dedup + FAIL only (day or night)
    반환: (df_1, df_2, df_3plus)
    각 DF 컬럼:
      prod_day, shift_type, pn, <bucket_col>, count, updated_at
    """
    now = datetime.now(tz=KST)

    # 빈 DF라도 스키마 유지
    if raw.empty:
        df1 = pd.DataFrame(columns=["prod_day","shift_type","pn","1회 FAIL_step_description","count","updated_at"])
        df2 = pd.DataFrame(columns=["prod_day","shift_type","pn","2회 반복_FAIL_step_description","count","updated_at"])
        df3 = pd.DataFrame(columns=["prod_day","shift_type","pn","3회 이상 반복_FAIL_step_description","count","updated_at"])
        return df1, df2, df3

    # (barcode, step_description) 단위로 발생 횟수(=반복 횟수)
    pair_cnt = (
        raw.groupby(["barcode_information", "step_description"], as_index=False)
           .size()
           .rename(columns={"size": "repeat_cnt"})
    )

    # pn은 barcode 기준으로 유일(가정) -> merge
    barcode_pn = raw[["barcode_information", "pn"]].drop_duplicates("barcode_information")
    pair_cnt = pair_cnt.merge(barcode_pn, on="barcode_information", how="left")

    def summarize(bucket: pd.DataFrame, colname: str) -> pd.DataFrame:
        if bucket.empty:
            return pd.DataFrame(columns=["prod_day","shift_type","pn",colname,"count","updated_at"])

        prod_day_val = raw["prod_day"].iloc[0]
        shift_val = raw["shift_type"].iloc[0]

        out = (
            bucket.groupby(["pn", "step_description"], as_index=False)
                  .agg(count=("barcode_information", "nunique"))
        )
        out.insert(0, "shift_type", shift_val)
        out.insert(0, "prod_day", prod_day_val)
        out.rename(columns={"step_description": colname}, inplace=True)
        out["updated_at"] = now
        return out

    b1 = pair_cnt[pair_cnt["repeat_cnt"] == 1]
    b2 = pair_cnt[pair_cnt["repeat_cnt"] == 2]
    b3 = pair_cnt[pair_cnt["repeat_cnt"] >= 3]

    df1 = summarize(b1, "1회 FAIL_step_description")
    df2 = summarize(b2, "2회 반복_FAIL_step_description")
    df3 = summarize(b3, "3회 이상 반복_FAIL_step_description")
    return df1, df2, df3

day_1, day_2, day_3p = build_repeat_dfs(day_raw)
night_1, night_2, night_3p = build_repeat_dfs(night_raw)

def sort_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    step_col = [c for c in df.columns if "step_description" in c][0]
    return df.sort_values(["pn", "count", step_col], ascending=[True, False, True]).reset_index(drop=True)

day_1 = sort_df(day_1)
day_2 = sort_df(day_2)
day_3p = sort_df(day_3p)
night_1 = sort_df(night_1)
night_2 = sort_df(night_2)
night_3p = sort_df(night_3p)

day_1.head()


,prod_day,shift_type,pn,1회 FAIL_step_description,count,updated_at
0,20260130,day,35930927,intensity7,8,2026-01-31 14:17:44.262556+09:00
1,20260130,day,35930927,intensity1,6,2026-01-31 14:17:44.262556+09:00
2,20260130,day,35930927,intensity2,3,2026-01-31 14:17:44.262556+09:00
3,20260130,day,35930927,intensity3,3,2026-01-31 14:17:44.262556+09:00
4,20260130,day,35930927,intensity4,3,2026-01-31 14:17:44.262556+09:00


In [5]:
# =========================
# 4) 결과 저장 (DB)
# =========================
# 요구사항:
# - DF를 i_daily_report 스키마의 지정 테이블에 저장
# - prod_day + shift_type(=day/night) 기준으로 기존 행은 삭제 후 재적재 (다른 날짜 데이터는 유지)

SAVE_SCHEMA = "i_daily_report"

TABLES = {
    ("day",   "1"): "d_vs_1time_step_decription_day_daily",
    ("day",   "2"): "d_vs_2time_step_decription_day_daily",
    ("day", "3+"): "d_vs_3time_over_step_decription_day_daily",
    ("night", "1"): "d_vs_1time_step_decription_night_daily",
    ("night", "2"): "d_vs_2time_step_decription_night_daily",
    ("night","3+"): "d_vs_3time_over_step_decription_night_daily",
}

def quote_ident(name: str) -> str:
    # double-quote and escape internal quotes
    return '"' + name.replace('"', '""') + '"'

def ensure_schema(engine: Engine, schema: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {quote_ident(schema)};"))

def ensure_table(engine: Engine, schema: str, table: str, columns: list[str]) -> None:
    # 모두 TEXT, updated_at만 timestamptz
    col_ddls = []
    for c in columns:
        if c == "updated_at":
            col_ddls.append(f"{quote_ident(c)} timestamptz")
        else:
            col_ddls.append(f"{quote_ident(c)} text")
    ddl = f"""
    CREATE TABLE IF NOT EXISTS {quote_ident(schema)}.{quote_ident(table)} (
      {", ".join(col_ddls)}
    );
    """
    with engine.begin() as conn:
        conn.execute(text(ddl))

def delete_partition(engine: Engine, schema: str, table: str, prod_day: str, shift_type: str) -> None:
    sql = f"""
    DELETE FROM {quote_ident(schema)}.{quote_ident(table)}
     WHERE {quote_ident("prod_day")} = :prod_day
       AND {quote_ident("shift_type")} = :shift_type;
    """
    with engine.begin() as conn:
        conn.execute(text(sql), {"prod_day": prod_day, "shift_type": shift_type})

def insert_df(engine: Engine, schema: str, table: str, df: pd.DataFrame) -> None:
    if df.empty:
        return

    cols = list(df.columns)
    col_sql = ", ".join(quote_ident(c) for c in cols)

    # bind param은 공백/한글 컬럼명 때문에 안전한 p0,p1...로 치환
    params = [f"p{i}" for i in range(len(cols))]
    val_sql = ", ".join(f":{p}" for p in params)

    sql = f"""
    INSERT INTO {quote_ident(schema)}.{quote_ident(table)} ({col_sql})
    VALUES ({val_sql});
    """

    records = df.to_dict(orient="records")
    payload = []
    for r in records:
        row = {}
        for i, c in enumerate(cols):
            row[params[i]] = r.get(c)
        payload.append(row)

    with engine.begin() as conn:
        conn.execute(text(sql), payload)

# --- 실행 ---
ensure_schema(engine, SAVE_SCHEMA)

to_save = [
    ("day",   "1",  day_1),
    ("day",   "2",  day_2),
    ("day",  "3+",  day_3p),
    ("night", "1",  night_1),
    ("night", "2",  night_2),
    ("night","3+",  night_3p),
]

for shift_type, bucket, df in to_save:
    table = TABLES[(shift_type, bucket)]
    ensure_table(engine, SAVE_SCHEMA, table, list(df.columns))
    # 재실행 시 해당 prod_day+shift_type만 교체(다른 날짜는 유지)
    delete_partition(engine, SAVE_SCHEMA, table, prod_day, shift_type)
    insert_df(engine, SAVE_SCHEMA, table, df)

print("[SAVE] done -> i_daily_report.* (day/night, 1/2/3+)")


[SAVE] done -> i_daily_report.* (day/night, 1/2/3+)


In [6]:
# =========================
# 5) 결과 출력 (사양: DataFrame만)
# =========================
print(f"[INPUT] prod_day={prod_day}")
print("[DAY] window =", shift_window(prod_day, "day"))
print("[NIGHT] window =", shift_window(prod_day, "night"))

print("\n===== DAY / 1회 =====")
display(day_1)

print("\n===== DAY / 2회 =====")
display(day_2)

print("\n===== DAY / 3회 이상 =====")
display(day_3p)

print("\n===== NIGHT / 1회 =====")
display(night_1)

print("\n===== NIGHT / 2회 =====")
display(night_2)

print("\n===== NIGHT / 3회 이상 =====")
display(night_3p)


[INPUT] prod_day=20260130
[DAY] window = (datetime.datetime(2026, 1, 30, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), datetime.datetime(2026, 1, 30, 20, 29, 59, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))
[NIGHT] window = (datetime.datetime(2026, 1, 30, 20, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), datetime.datetime(2026, 1, 31, 8, 29, 59, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))

===== DAY / 1회 =====


,prod_day,shift_type,pn,1회 FAIL_step_description,count,updated_at
0,20260130,day,35930927,intensity7,8,2026-01-31 14:17:44.262556+09:00
1,20260130,day,35930927,intensity1,6,2026-01-31 14:17:44.262556+09:00
2,20260130,day,35930927,intensity2,3,2026-01-31 14:17:44.262556+09:00
3,20260130,day,35930927,intensity3,3,2026-01-31 14:17:44.262556+09:00
4,20260130,day,35930927,intensity4,3,2026-01-31 14:17:44.262556+09:00
5,20260130,day,35930927,intensity5,3,2026-01-31 14:17:44.262556+09:00
6,20260130,day,35930927,intensity6,3,2026-01-31 14:17:44.262556+09:00
7,20260130,day,35930927,intensity8,3,2026-01-31 14:17:44.262556+09:00
8,20260130,day,35930928,intensity1,2,2026-01-31 14:17:44.262556+09:00
9,20260130,day,35930928,intensity2,2,2026-01-31 14:17:44.262556+09:00



===== DAY / 2회 =====


,prod_day,shift_type,pn,2회 반복_FAIL_step_description,count,updated_at



===== DAY / 3회 이상 =====


...

,prod_day,shift_type,pn,1회 FAIL_step_description,count,updated_at
0,20260130,night,35930928,intensity7,9,2026-01-31 14:17:44.289914+09:00
1,20260130,night,35930928,intensity1,8,2026-01-31 14:17:44.289914+09:00



===== NIGHT / 2회 =====


,prod_day,shift_type,pn,2회 반복_FAIL_step_description,count,updated_at



===== NIGHT / 3회 이상 =====


,prod_day,shift_type,pn,3회 이상 반복_FAIL_step_description,count,updated_at


## 참고
- `result='FAIL'`만 조회합니다.
- `end_day`, `end_time`이 `"YYYYMMDD"`, `"HH:MI:SS"` 형태의 text라는 전제에서 문자열 비교로 필터링합니다.
- 중복 제거는 SQL에서 `DISTINCT ON (barcode_information, step_description, end_day, end_time)`로 수행합니다.
- pn 매핑 실패(키 없음/18자리 미만 등)는 `Unknown`으로 처리합니다.
